In [1]:
import rospy
from assignment2_rt_part1.msg import RobotState, Target
from nav_msgs.msg import Odometry
from assignment_2_2024.msg import PlanningAction, PlanningGoal
import actionlib
import threading
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# Global variables
robot_state = RobotState()
client = None
stop_event = threading.Event()

# Publisher
target_pub = None

# Widget for visualizing the state of the robot
robot_state_display = widgets.Label()

In [2]:
# Function to receive odometry data and update position and velocity
def odom_callback(msg):
    global robot_state
    # Get position and velocity from odometry data
    robot_state.x = msg.pose.pose.position.x
    robot_state.y = msg.pose.pose.position.y
    robot_state.vel_x = msg.twist.twist.linear.x
    robot_state.vel_z = msg.twist.twist.angular.z

In [3]:
# Function to update the visualization ot the position and the velocity of the robot in real time
def update_display():
    while not stop_event.is_set():
        robot_state_display.value = f"Position: (x: {robot_state.x:.2f}, y: {robot_state.y:.2f}) |  Velocity: lin {robot_state.vel_x:.2f}, ang {robot_state.vel_z:.2f}"
        time.sleep(0.5)

In [4]:
def setup_ros():
    global client, target_pub

    rospy.init_node('jupyter_interface_node', anonymous=True)
    
    # Subscriber
    rospy.Subscriber('/odom', Odometry, odom_callback)
    # Publisher
    target_pub = rospy.Publisher('target_topic', Target, queue_size=10)

    client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
    rospy.loginfo("Waiting for action server to start...")
    client.wait_for_server()
    rospy.loginfo("Action server started.")

    display(robot_state_display)

    # Thread to update the state
    t = threading.Thread(target=update_display)
    t.daemon = True
    t.start()


In [5]:
# Widget input
x_input = widgets.FloatText(description='Target X:')
y_input = widgets.FloatText(description='Target Y:')
send_button = widgets.Button(description='Send Goal', button_style='success')
cancel_button = widgets.Button(description='Cancel Goal', button_style='danger')
output_area = widgets.Output()

# Function to send a goal to the action server
def send_goal_clicked(b):
    with output_area:
        clear_output()
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x_input.value
        goal.target_pose.pose.position.y = y_input.value
        client.send_goal(goal)

        target_msg = Target(x=x_input.value, y=y_input.value)
        target_pub.publish(target_msg)

        print(f" Goal sent: ({x_input.value}, {y_input.value})")

# Function to cancel a sent goal
def cancel_goal_clicked(b):
    with output_area:
        clear_output()
        client.cancel_goal()
        print(" Stopping the robot.")

send_button.on_click(send_goal_clicked)
cancel_button.on_click(cancel_goal_clicked)

ui = widgets.VBox([
    widgets.HBox([x_input, y_input]),
    widgets.HBox([send_button, cancel_button]),
    output_area
])

def main():
    setup_ros()
    display(ui)


In [6]:
main()


[INFO] [1744962905.243539, 1249.908000]: Aspettando il server di azione...
[INFO] [1744962905.337703, 1249.986000]: Server di azione pronto.


Label(value='')